# Phase 4 — Métriques de classification : « panne dans les N prochains cycles ? »

Application de la **section 4** de [`docs/protocole_evaluation.md`](../docs/protocole_evaluation.md) au XGBoost de la Phase 3 (FD001).

- **Étiquette** : positif (1) si RUL vraie ≤ N, négatif (0) sinon. N n'est pas choisi : on calcule pour N = 15, 30, 50.
- **Deux approches comparées** : (1) alerte dérivée de la régression (RUL prédite ≤ N, score `-RUL prédite`) ; (2) classifieur dédié (`LogisticRegression`, `XGBClassifier`).
- **Règle** : le seuil se choisit sur la validation (rappel minimal fixé d'abord), jamais sur le test. Le test officiel n'est lu que dans le tableau final.

In [ ]:
import sys
sys.path.insert(0, "..")  # pour importer src/ depuis notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import xgboost

from src.preprocessing import (
    load_fd001, get_sensor_columns, find_constant_sensors,
    normalize_minmax, lisser_capteurs, calculer_rul, plafonner_rul, DATA_DIR,
)

FENETRE = 5         # fenetre glissante (Phase 1/3)
PLAFOND_RUL = 125   # plafond de la RUL (Phase 2)
VALEURS_N = [15, 30, 50]  # les trois horizons testes, N n'etant pas encore choisi par l'equipe
GRAINE = 42

print(f"scikit-learn {sklearn.__version__}, xgboost {xgboost.__version__}, numpy {np.__version__}, pandas {pd.__version__}")

In [ ]:
def ajouter_features_glissantes(df, sensor_cols, fenetre):
    """
    Ajoute, pour chaque capteur, la moyenne et l'ecart-type glissants sur `fenetre` cycles, moteur par moteur
    (meme fonction que dans notebooks/03_modelisation.ipynb, recopiee a l'identique).

    Parameters
    ----------
    df : pandas.DataFrame
        Tableau source (une copie est renvoyee).
    sensor_cols : list de str
        Capteurs pour lesquels calculer les features.
    fenetre : int
        Taille de la fenetre glissante (causale).

    Returns
    -------
    pandas.DataFrame
        Tableau avec "<capteur>_glissante_moy" et "<capteur>_glissante_std" en plus.
    """
    df = df.copy()  # copie independante
    for sensor_name in sensor_cols:  # un capteur a la fois
        colonne_moyenne = sensor_name + "_glissante_moy"
        colonne_ecart_type = sensor_name + "_glissante_std"
        df[colonne_moyenne] = 0.0
        df[colonne_ecart_type] = 0.0
        for moteur_id in df["unit_number"].unique():  # un moteur a la fois : jamais de melange entre moteurs
            masque = df["unit_number"] == moteur_id
            df.loc[masque, colonne_moyenne] = df.loc[masque, sensor_name].rolling(fenetre, min_periods=1).mean()
            df.loc[masque, colonne_ecart_type] = df.loc[masque, sensor_name].rolling(fenetre, min_periods=1).std()
    return df

In [ ]:
# --- Train : meme chaine que le notebook 03 (capteurs constants, min-max, lissage, RUL cappee, features) ---
train = load_fd001("train")
sensor_cols = get_sensor_columns(train.columns)
constant_sensors = find_constant_sensors(train, sensor_cols)
train = train.drop(columns=constant_sensors)
sensor_cols_restants = get_sensor_columns(train.columns)

train, scaler = normalize_minmax(train, sensor_cols_restants)
train = lisser_capteurs(train, sensor_cols_restants, fenetre=FENETRE)
train = calculer_rul(train)
train = plafonner_rul(train, plafond=PLAFOND_RUL)
train = ajouter_features_glissantes(train, sensor_cols_restants, FENETRE)

# noms des colonnes glissantes, par boucle explicite
colonnes_moyenne = []
colonnes_ecart_type = []
for colonne in train.columns:
    if colonne.endswith("_glissante_moy"):
        colonnes_moyenne.append(colonne)
    if colonne.endswith("_glissante_std"):
        colonnes_ecart_type.append(colonne)
train[colonnes_ecart_type] = train[colonnes_ecart_type].fillna(0.0)  # 1er cycle : ecart-type indefini -> 0

feature_cols = sensor_cols_restants + colonnes_moyenne + colonnes_ecart_type + ["time_cycles"]
X = train[feature_cols]
y_rul = train["RUL_cappee"]        # RUL plafonnee a 125 (cible de la regression)
groups = train["unit_number"]      # un identifiant de moteur par ligne, pour ne jamais separer un moteur
assert len(feature_cols) == 46, f"attendu 46 features, trouve {len(feature_cols)}"
print(f"X : {X.shape}, {groups.nunique()} moteurs")

In [ ]:
# --- Test officiel : une ligne par moteur (dernier cycle), RUL vraie NON plafonnee ---
# Ces objets sont prepares ici mais NE SERVENT PAS a choisir quoi que ce soit : lus uniquement dans le tableau final.
test = load_fd001("test")
test = test.drop(columns=constant_sensors)
test[sensor_cols_restants] = scaler.transform(test[sensor_cols_restants])  # transform seul : scaler appris sur le train
test = lisser_capteurs(test, sensor_cols_restants, fenetre=FENETRE)
test = ajouter_features_glissantes(test, sensor_cols_restants, FENETRE)
test[colonnes_ecart_type] = test[colonnes_ecart_type].fillna(0.0)

indices_derniers_cycles = test.groupby("unit_number")["time_cycles"].idxmax()
dernier_cycle_par_moteur = test.loc[indices_derniers_cycles].sort_values("unit_number").reset_index(drop=True)
rul_test_vraie = pd.read_csv(f"{DATA_DIR}/RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL_vraie"])
X_test = dernier_cycle_par_moteur[feature_cols]
y_test_rul = rul_test_vraie["RUL_vraie"].values
assert X_test.shape[0] == 100 and len(y_test_rul) == 100
print(f"Test : {X_test.shape}")

## 1. Étiquettes « RUL ≤ N » et proportion de positifs

Piège à anticiper : la validation contient **tous les cycles** de chaque moteur (beaucoup de cycles « en bonne santé »), alors que le test ne contient **que le dernier cycle observé** de chaque moteur. La proportion de positifs n'est donc pas la même des deux côtés.

In [ ]:
def fabriquer_etiquettes(rul, n):
    """
    Transforme une RUL (en cycles) en etiquette binaire : 1 si RUL <= n (moteur a retirer), 0 sinon.

    Parameters
    ----------
    rul : array-like
        Les valeurs de RUL (vraies).
    n : int
        L'horizon d'alerte, en cycles.

    Returns
    -------
    numpy.ndarray
        Tableau d'entiers 0/1, de meme longueur que `rul`.
    """
    etiquettes = []  # une etiquette par valeur de RUL
    for valeur in rul:
        if valeur <= n:  # panne dans les n prochains cycles
            etiquettes.append(1)
        else:
            etiquettes.append(0)
    return np.array(etiquettes)

# sanity check sur des valeurs ou la reponse est evidente
assert list(fabriquer_etiquettes([0, 15, 16, 125], 15)) == [1, 1, 0, 0]

print("N    | train/validation (20631 lignes) | test officiel (100 moteurs)")
for n in VALEURS_N:
    etiq_train = fabriquer_etiquettes(y_rul.values, n)
    etiq_test = fabriquer_etiquettes(y_test_rul, n)
    print(f"{n:<4d} | {etiq_train.sum():6d} positifs = {100 * etiq_train.mean():5.1f} %      | {etiq_test.sum():4d} positifs = {100 * etiq_test.mean():5.1f} %")

## 2. Plis par moteur (un jeu par N) et prédictions hors-pli de la régression

`StratifiedGroupKFold` garde les moteurs groupés **et** répartit les positifs de façon équilibrée entre les plis. Comme l'étiquette dépend de N, il y a **un jeu de plis par N**. Ces plis sont fixés une fois et réutilisés par toutes les approches (régression dérivée, `LogisticRegression`, `XGBClassifier`), conformément au §1 du protocole.

Pour l'approche 1 (alerte dérivée de la régression), on réentraîne le XGBoost de la Phase 3 (`n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42`) sur chaque pli, et on garde pour chaque ligne de validation la **prédiction hors-pli** : celle d'un modèle qui n'a jamais vu ce moteur.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from xgboost import XGBRegressor

etiquettes_par_n = {}  # n -> tableau 0/1 de la validation
plis_par_n = {}        # n -> liste de 5 couples (indices_train, indices_val)

for n in VALEURS_N:
    etiquettes = fabriquer_etiquettes(y_rul.values, n)
    etiquettes_par_n[n] = etiquettes

    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=GRAINE)
    plis = list(sgkf.split(X, etiquettes, groups=groups))  # calcule UNE fois, reutilise partout
    plis_par_n[n] = plis

    print(f"N={n} :")
    for i, (idx_train, idx_val) in enumerate(plis):
        # verification explicite : aucun moteur des deux cotes du pli
        moteurs_train = set(groups.iloc[idx_train])
        moteurs_val = set(groups.iloc[idx_val])
        assert len(moteurs_train & moteurs_val) == 0, "un moteur est present des deux cotes !"
        part_positifs = 100 * etiquettes[idx_val].mean()
        print(f"   pli {i + 1} : {len(moteurs_val)} moteurs en validation, {etiquettes[idx_val].sum():4d} positifs = {part_positifs:4.1f} % (global {100 * etiquettes.mean():.1f} %)")

In [ ]:
# Predictions hors-pli de la regression, pour chaque N (modele tout neuf a chaque pli)
rul_predite_par_n = {}  # n -> RUL predite pour chaque ligne de validation (hors-pli)
pli_de_chaque_ligne_par_n = {}  # n -> numero du pli de validation de chaque ligne

for n in VALEURS_N:
    predictions = np.zeros(len(X))  # une prediction par ligne, remplie pli par pli
    numero_pli = np.zeros(len(X), dtype=int)

    for i, (idx_train, idx_val) in enumerate(plis_par_n[n]):
        modele = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=GRAINE)
        modele.fit(X.iloc[idx_train], y_rul.iloc[idx_train])  # apprend sur les moteurs d'entrainement du pli
        predictions[idx_val] = modele.predict(X.iloc[idx_val])  # predit des moteurs jamais vus
        numero_pli[idx_val] = i

    rul_predite_par_n[n] = predictions
    pli_de_chaque_ligne_par_n[n] = numero_pli

    # sanity check : RMSE hors-pli, a comparer au ~16.9 de la Phase 3 (autres plis, donc pas identique)
    rmse = np.sqrt(np.mean((y_rul.values - predictions) ** 2))
    print(f"N={n} : RMSE hors-pli de la regression = {rmse:.2f} (Phase 3 : 16.86 sur d'autres plis)")
    assert 14 < rmse < 20, "RMSE hors-pli incoherent avec la Phase 3, a verifier"

## 3. Approche 1 — alerte dérivée de la régression

Décision : alerte (1) si **RUL prédite ≤ seuil**. Score continu pour la ROC / précision-rappel : `-RUL prédite` (plus la RUL prédite est basse, plus le moteur est urgent).

Point de méthode : le **seuil de décision n'a pas de raison d'être égal à N**. Prendre « RUL prédite ≤ N » est un choix particulier ; le seuil sera choisi sur la validation (rappel minimal fixé d'abord). On regarde d'abord ce que donne le seuil naïf = N.

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

def metriques_a_seuil(y_vrai, score, seuil):
    """
    Calcule les metriques de classification pour une regle 'alerte si score >= seuil'.

    Parameters
    ----------
    y_vrai : array-like de 0/1
        Les vraies etiquettes.
    score : array-like de float
        Score continu : plus il est grand, plus le moteur est juge urgent.
    seuil : float
        Seuil de decision : alerte (1) si score >= seuil.

    Returns
    -------
    dict
        Matrice de confusion (tn, fp, fn, tp), accuracy, precision, rappel, F1.
    """
    prediction = []  # une decision 0/1 par ligne
    for valeur in score:
        if valeur >= seuil:
            prediction.append(1)
        else:
            prediction.append(0)
    prediction = np.array(prediction)

    tn, fp, fn, tp = confusion_matrix(y_vrai, prediction, labels=[0, 1]).ravel()  # lignes = vrai, colonnes = predit
    resultat = {
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "accuracy": accuracy_score(y_vrai, prediction),
        "precision": precision_score(y_vrai, prediction, zero_division=0),
        "rappel": recall_score(y_vrai, prediction, zero_division=0),
        "f1": f1_score(y_vrai, prediction, zero_division=0),
    }
    return resultat

print("Approche 1, seuil naif 'RUL predite <= N' (score = -RUL predite, donc seuil = -N), validation hors-pli :")
print(f"{'N':>3s} | {'positifs':>8s} | {'accuracy':>8s} | {'precision':>9s} | {'rappel':>6s} | {'F1':>5s} | {'AUC':>5s} | {'AP':>5s} |  tn    fp   fn    tp")
for n in VALEURS_N:
    score = -rul_predite_par_n[n]  # plus la RUL predite est basse, plus le score est haut
    m = metriques_a_seuil(etiquettes_par_n[n], score, -n)
    auc = roc_auc_score(etiquettes_par_n[n], score)
    ap = average_precision_score(etiquettes_par_n[n], score)
    print(f"{n:3d} | {100 * etiquettes_par_n[n].mean():7.1f}% | {m['accuracy']:8.3f} | {m['precision']:9.3f} | {m['rappel']:6.3f} | {m['f1']:5.3f} | {auc:5.3f} | {ap:5.3f} | {m['tn']:5d} {m['fp']:5d} {m['fn']:4d} {m['tp']:5d}")

## 4. Approche 2 — classifieurs dédiés, et baseline aléatoire

- **`LogisticRegression`** : modèle linéaire sur les 46 features. Comme elle est sensible à l'échelle des variables, on l'enveloppe dans un `Pipeline` avec un `StandardScaler` : le scaler est ré-appris **dans chaque pli**, sur l'entraînement seul (§1 point 3, aucune fuite).
- **`XGBClassifier`** : mêmes réglages que le régresseur (`n_estimators=200, max_depth=5, learning_rate=0.05`), objectif de classification binaire.
- **Baseline aléatoire** (§2) : un tirage uniforme entre 0 et 125 (graine fixe) tient lieu de « RUL prédite ». C'est le plancher : AUC ≈ 0,5.

Aucun rééquilibrage des classes (pas de `class_weight`, pas de `scale_pos_weight`) : on garde les réglages par défaut pour avoir un point de comparaison honnête, et on traite le déséquilibre par le **choix du seuil** (section suivante).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

# scores_par_n[n][nom_approche] = score continu hors-pli pour chaque ligne de validation
# (plus le score est haut, plus le moteur est juge urgent)
scores_par_n = {}

for n in VALEURS_N:
    etiquettes = etiquettes_par_n[n]
    scores = {}

    # approche 1 : on reprend la regression de la section 2, score = -RUL predite
    scores["Regression XGBoost (-RUL predite)"] = -rul_predite_par_n[n]

    # approche 2a et 2b : classifieurs entraines sur l'etiquette 0/1, predict_proba = probabilite d'etre positif
    proba_logistique = np.zeros(len(X))
    proba_xgb = np.zeros(len(X))
    for idx_train, idx_val in plis_par_n[n]:  # les memes 5 plis que la regression
        modele_log = Pipeline([
            ("scaler", StandardScaler()),  # appris sur l'entrainement du pli uniquement
            ("logistique", LogisticRegression(max_iter=2000, random_state=GRAINE)),
        ])
        modele_log.fit(X.iloc[idx_train], etiquettes[idx_train])
        proba_logistique[idx_val] = modele_log.predict_proba(X.iloc[idx_val])[:, 1]  # colonne 1 = classe positive

        modele_xgb = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=GRAINE)
        modele_xgb.fit(X.iloc[idx_train], etiquettes[idx_train])
        proba_xgb[idx_val] = modele_xgb.predict_proba(X.iloc[idx_val])[:, 1]

    scores["LogisticRegression"] = proba_logistique
    scores["XGBClassifier"] = proba_xgb

    # baseline aleatoire : RUL "predite" tiree uniformement entre 0 et 125, graine fixe
    generateur = np.random.default_rng(GRAINE)
    rul_aleatoire = generateur.uniform(0, PLAFOND_RUL, size=len(X))
    scores["Aleatoire (baseline)"] = -rul_aleatoire

    scores_par_n[n] = scores

# tableau : AUC et precision moyenne, qui ne dependent d'aucun seuil
print("AUC ROC / precision moyenne (AP), validation hors-pli. Reference AP du hasard = proportion de positifs.")
for n in VALEURS_N:
    print(f"\nN={n} (positifs : {100 * etiquettes_par_n[n].mean():.1f} %)")
    for nom, score in scores_par_n[n].items():
        auc = roc_auc_score(etiquettes_par_n[n], score)
        ap = average_precision_score(etiquettes_par_n[n], score)
        print(f"   {nom:36s} AUC={auc:.3f}   AP={ap:.3f}")
    auc_alea = roc_auc_score(etiquettes_par_n[n], scores_par_n[n]["Aleatoire (baseline)"])
    assert abs(auc_alea - 0.5) < 0.03, "la baseline aleatoire devrait avoir une AUC proche de 0,5"

In [ ]:
# Au seuil "par defaut" des classifieurs (probabilite >= 0,5), pour montrer pourquoi ce seuil n'est pas adapte.
print("Classifieurs au seuil par defaut 0,5 (validation hors-pli) :")
print(f"{'N':>3s} | {'modele':18s} | {'accuracy':>8s} | {'precision':>9s} | {'rappel':>6s} | {'F1':>5s}")
for n in VALEURS_N:
    for nom in ["LogisticRegression", "XGBClassifier"]:
        m = metriques_a_seuil(etiquettes_par_n[n], scores_par_n[n][nom], 0.5)
        print(f"{n:3d} | {nom:18s} | {m['accuracy']:8.3f} | {m['precision']:9.3f} | {m['rappel']:6.3f} | {m['f1']:5.3f}")

## 5. Choix du seuil de décision (sur la validation uniquement)

**Règle écrite avant de regarder les résultats** (protocole §4) :

1. On fixe un **rappel minimal** à atteindre. Pourquoi le rappel d'abord ? Les deux erreurs ne coûtent pas la même chose : un **faux négatif** (moteur proche de la panne laissé en vol) est le cas grave, un **faux positif** (moteur retiré trop tôt) coûte une inspection ou une immobilisation inutile. On accepte donc de perdre en précision pour ne pas manquer de moteurs.
2. Parmi tous les seuils qui atteignent ce rappel, on retient celui qui **maximise la précision** (c'est-à-dire le moins de fausses alertes possible). À précision égale, on prend le seuil le plus sévère (le moins d'alertes).
3. Rappel cible principal : **0,95**. On montre aussi 0,90 et 0,99 pour que l'équipe voie ce que coûte chaque exigence de sécurité : ce chiffre est une décision de maintenance, pas une décision statistique.

Trois vérifications pour que le choix soit défendable : (a) sensibilité au rappel visé, (b) le seuil **tient-il sur des moteurs jamais vus ?** (choisi sur 4 plis, évalué sur le 5e), (c) pour l'approche régression, le seuil est traduit en **cycles**.

In [ ]:
from sklearn.metrics import precision_recall_curve

def choisir_seuil(y_vrai, score, rappel_min):
    """
    Choisit le seuil de decision : parmi les seuils dont le rappel est >= rappel_min,
    celui qui maximise la precision ; a precision egale, le plus severe (le plus haut).

    Parameters
    ----------
    y_vrai : array-like de 0/1
        Les vraies etiquettes (du jeu sur lequel on choisit, jamais le test).
    score : array-like de float
        Score continu (plus il est haut, plus le moteur est urgent).
    rappel_min : float
        Rappel minimal exige (entre 0 et 1).

    Returns
    -------
    float
        Le seuil retenu : alerte si score >= seuil.
    """
    precisions, rappels, seuils = precision_recall_curve(y_vrai, score)
    # precision_recall_curve renvoie une precision et un rappel de plus que de seuils
    # (le point "aucune alerte") : on ne regarde que les points qui correspondent a un seuil
    meilleur_seuil = None
    meilleure_precision = -1.0
    for i in range(len(seuils)):  # un candidat de seuil a la fois
        if rappels[i] >= rappel_min:  # le seuil respecte la contrainte de securite
            if precisions[i] > meilleure_precision or (precisions[i] == meilleure_precision and seuils[i] > meilleur_seuil):
                meilleure_precision = precisions[i]
                meilleur_seuil = seuils[i]
    return meilleur_seuil

# sanity check sur un exemple jouet ou la reponse se calcule a la main :
# scores 0.9, 0.8, 0.4, 0.3 avec vraies etiquettes 1, 1, 0, 1 -> pour un rappel >= 1, il faut descendre jusqu'a 0.3
assert choisir_seuil([1, 1, 0, 1], [0.9, 0.8, 0.4, 0.3], 1.0) == 0.3
assert choisir_seuil([1, 1, 0, 1], [0.9, 0.8, 0.4, 0.3], 0.6) == 0.8  # 2 positifs sur 3 detectes, precision 1

In [ ]:
APPROCHES = ["Regression XGBoost (-RUL predite)", "LogisticRegression", "XGBClassifier"]
RAPPELS_CIBLES = [0.90, 0.95, 0.99]

print("(a) Sensibilite au rappel vise - seuil choisi sur toute la validation hors-pli")
for n in VALEURS_N:
    print(f"\nN={n}")
    print(f"   {'approche':34s} | {'rappel vise':>11s} | {'seuil':>16s} | {'precision':>9s} | {'rappel':>6s} | {'FP':>5s} | {'FN':>5s}")
    for nom in APPROCHES:
        score = scores_par_n[n][nom]
        for rappel_cible in RAPPELS_CIBLES:
            seuil = choisir_seuil(etiquettes_par_n[n], score, rappel_cible)
            m = metriques_a_seuil(etiquettes_par_n[n], score, seuil)
            if nom.startswith("Regression"):
                texte_seuil = f"RUL pred <= {-seuil:5.1f}"  # score = -RUL predite : on revient en cycles
            else:
                texte_seuil = f"proba >= {seuil:6.3f}"
            print(f"   {nom:34s} | {rappel_cible:11.2f} | {texte_seuil:>16s} | {m['precision']:9.3f} | {m['rappel']:6.3f} | {m['fp']:5d} | {m['fn']:5d}")

In [ ]:
# (b) Le seuil tient-il sur des moteurs jamais vus ? Pour chaque pli k : seuil choisi sur les 4 AUTRES plis,
# puis applique au pli k. On regarde le rappel obtenu sur le pli k, a comparer au rappel vise.
RAPPEL_PRINCIPAL = 0.95
print(f"(b) Generalisation du seuil (rappel vise {RAPPEL_PRINCIPAL}) : seuil choisi sur 4 plis, evalue sur le 5e")
print("    (note : les scores des 4 plis de choix viennent de modeles qui ont vu les moteurs du pli evalue ;")
print("     le biais est faible car seul un seuil, un nombre, est transfere, mais il n'est pas nul)\n")

resume_generalisation = {}  # (n, approche) -> (liste des rappels, liste des precisions, liste des seuils)
for n in VALEURS_N:
    numero_pli = pli_de_chaque_ligne_par_n[n]
    for nom in APPROCHES:
        score = scores_par_n[n][nom]
        rappels_plis = []
        precisions_plis = []
        seuils_plis = []
        for k in range(5):
            masque_choix = numero_pli != k   # les 4 autres plis
            masque_eval = numero_pli == k    # le pli tenu a part
            seuil_k = choisir_seuil(etiquettes_par_n[n][masque_choix], score[masque_choix], RAPPEL_PRINCIPAL)
            m = metriques_a_seuil(etiquettes_par_n[n][masque_eval], score[masque_eval], seuil_k)
            rappels_plis.append(m["rappel"])
            precisions_plis.append(m["precision"])
            seuils_plis.append(seuil_k)
        resume_generalisation[(n, nom)] = (rappels_plis, precisions_plis, seuils_plis)
        if nom.startswith("Regression"):
            seuils_affiches = -np.array(seuils_plis)  # en cycles
            texte_seuils = f"RUL pred <= {seuils_affiches.mean():5.1f} (min {seuils_affiches.min():.1f}, max {seuils_affiches.max():.1f})"
        else:
            texte_seuils = f"proba >= {np.mean(seuils_plis):.3f} (min {np.min(seuils_plis):.3f}, max {np.max(seuils_plis):.3f})"
        print(f"N={n:2d} {nom:34s} rappel sur pli tenu a part = {np.mean(rappels_plis):.3f} +/- {np.std(rappels_plis):.3f} "
              f"(min {np.min(rappels_plis):.3f}) | precision = {np.mean(precisions_plis):.3f} +/- {np.std(precisions_plis):.3f} | {texte_seuils}")